In [ ]:
"""
ROGII Wellbore Geology Prediction â€” v12
DSC 204A Final Project

Strategy:
  - Visible training wells: physical model (RMSE ~0.007 ft)
  - Hidden test wells: PF ensemble ONLY â€” 128 seeds, lik-weighted (scale=5)
    init_spread=2.0 ft (wider initial particle spread)
    GR interpolated before PF (fills NaN gaps so PF always has observations)
    Local avg: 4.71 ft (vs 5.95 ft without GR interpolation)
    Key insight: 000d7d20 has 47% NaN GR in prediction â€” interpolation critical
"""

import os, glob, warnings, json
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

SELECTOR_N_EVAL_THRESHOLD = 4840.0
SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)
SELECTOR_BIN_VARIANTS = {
    0: "pf_scale_5_hold_0.2",
    1: "pf_scale_3_hold_0.15",
    2: "pf_scale_12_beam_0.2_hold_0.15",
    3: "pf_scale_5_hold_0.15",
    4: "pf_scale_5_beam_0.05_hold_0.05",
    5: "pf_scale_12_beam_0.2_hold_0.05",
}
SELECTOR_GLOBAL_VARIANT = "pf_scale_8_hold_0.2"
SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)
RESIDUAL_FEATURE_FILE = 'public_sel15_pf_oof_features.csv.gz'
RESIDUAL_RIDGE_ALPHA = 20.0
RESIDUAL_TARGET_CLIP = 80.0
RESIDUAL_PRED_CLIP = 20.0
RESIDUAL_SHRINK = 0.50
RESIDUAL_MAX_TRAIN_ROWS = 500_000
RESIDUAL_MAX_TRAIN_ROWS_PER_BUCKET = 120_000
RESIDUAL_RANDOM_SEED = 42
RESIDUAL_FEATURE_COLUMNS = [
    'pf_pred',
    'last_anchor_tvt',
    'beam_pred',
    'pf_selected_scale_pred',
    'pf_scale_3',
    'pf_scale_5',
    'pf_scale_8',
    'pf_scale_12',
    'pf_seed_mean',
    'pf_seed_std',
    'pf_lik_best',
    'pf_lik_mean',
    'pf_lik_std',
    'pf_lik_gap_best_second',
    'pf_weight_entropy',
    'pf_weight_top',
    'pf_weight_best_second_gap',
    'pf_effective_seeds',
    'pf_pred_minus_last_anchor',
    'pf_beam_diff',
    'abs_pf_beam_diff',
    'beam_spread',
    'beam_min',
    'beam_max',
    'beam_final_cost_mean',
    'beam_final_cost_min',
    'selector_code',
    'selector_scale',
    'selector_beam_weight',
    'selector_hold_weight',
    'selector_n_eval',
    'selector_z_span',
    'pseudo_cutoff_fraction',
    'cutoff_row',
    'row_idx',
    'prefix_length',
    'eval_step',
    'eval_fraction',
    'MD',
    'X',
    'Y',
    'Z',
    'GR',
    'gr_isna',
    'gr_prefix_availability',
    'gr_eval_availability',
    'pf_gr_sigma_mean',
    'pf_initial_rate_mean',
    'pf_mean_effective_particles',
    'pf_min_effective_particles',
    'pf_resample_count_mean',
]


def find_input_dir():
    for c in ['/kaggle/input/rogii-wellbore-geology-prediction',
              '/kaggle/input/competitions/rogii-wellbore-geology-prediction']:
        if os.path.isdir(c):
            print(f'INPUT_DIR={c}')
            return c
    hits = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
    if hits:
        d = os.path.dirname(hits[0])
        print(f'Discovered INPUT_DIR={d}')
        return d
    raise FileNotFoundError('Cannot locate competition data')


INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, 'train')
TEST_DIR  = os.path.join(INPUT_DIR, 'test')

_hw_files  = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
TEST_WELLS = [os.path.basename(f).split('__')[0] for f in _hw_files]
print(f'Test wells: {TEST_WELLS}')


def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw


def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
    tw_g = tw_tr.dropna(subset=['Geology'])
    ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    if np.isnan(ref_tvt):
        ref_col = tw_g['Geology'].iloc[0]
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
    return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset


def run_particle_filter(hw, tw, n_particles=500, seed=42):
    """Conservative PF. Returns (predictions_array, total_log_likelihood)."""
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input'])
    last_Z   = float(last['Z'])
    last_MD  = float(last['MD'])

    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    N   = n_particles
    rng = np.random.default_rng(seed)
    ls   = last_tvt + last_Z
    pos  = ls + 3.0 * rng.standard_normal(N)  # wider init spread helps wells with abrupt TVT shift at PS
    rate = ir + 0.01 * rng.standard_normal(N)
    w    = np.ones(N) / N

    MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5

    md_v = ev['MD'].values.astype(float)
    z_v  = ev['Z'].values.astype(float)
    # Interpolate GR gaps before tracking â€” critical for wells with high NaN fraction
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    gr_v = gr_interp.values.astype(float)[ev.index]

    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev))
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos   = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.exp(-0.5 * np.minimum(d**2, 600.))
        lk = np.maximum(lk, 1e-300)
        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        n_eff = 1.0 / (w**2).sum()
        if n_eff < RESAMP * N:
            cum = np.cumsum(w)
            u0  = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos  = pos[idx]  + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w    = np.ones(N) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    """
    128-seed lik-weighted PF ensemble.
    More seeds â†’ better coverage of the TVT exploration space.
    """
    preds = []
    liks  = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)

    liks   = np.array(liks)
    liks_n = liks - liks.max()
    weights = np.exp(liks_n / scale)
    weights /= weights.sum()

    return (weights[:, None] * np.stack(preds, 0)).sum(0)


def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
    preds = []
    liks = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)
    pred_arr = np.stack(preds, 0)
    liks = np.array(liks)
    liks_n = liks - liks.max()
    out = {}
    for scale in scales:
        weights = np.exp(liks_n / float(scale))
        weights /= weights.sum()
        out[f"pf_scale_{scale:g}"] = (weights[:, None] * pred_arr).sum(0)
    out["pf_mean"] = pred_arr.mean(0)
    return out


# 14 beam configs: original 7 + 7 new ones exploring broader parameter space
BEAM_CONFIGS = [
    # Original 7 configs (from ajayrao43)
    (10, 20.0, 144.0, 2),
    (10,  8.0,  64.0, 2),
    ( 8, 35.0, 220.0, 1),
    (10, 14.0,  90.0, 5),
    (20,  4.0,  36.0, 3),
    (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
    # 7 new configs: wider beam, different motion/error scales
    (20, 30.0, 200.0, 2),
    (15, 10.0,  80.0, 4),
    (25,  6.0,  50.0, 3),
    (10, 40.0, 300.0, 1),
    (12, 18.0, 120.0, 5),
    (30,  8.0,  70.0, 2),
    (10, 50.0, 400.0, 0),
]


def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    """Vectorized beam search for TVT tracking via GR matching."""
    n  = len(hgr)
    nt = len(tw_tvt)
    if n == 0:
        return np.array([last_tvt])

    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()

    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    MC    = mc * np.array([2., 1., 0., 1., 2.])

    bidx  = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf)
    bcost[0] = 0.
    bn = 1

    result = np.zeros(n)

    for step in range(n):
        gv = sgr[step]
        ni = bidx[:bn, None] + MOVES[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)

        gr_e = (gv - tw_gr[ci])**2 / es
        tot  = bcost[:bn, None] + gr_e + MC[None, :]
        tot  = np.where(valid, tot, np.inf)

        ni_f  = ni.flatten()
        tot_f = tot.flatten()
        vf    = valid.flatten()
        ni_f  = ni_f[vf]
        tot_f = tot_f[vf]

        order = np.argsort(tot_f)
        ni_s  = ni_f[order]
        tot_s = tot_f[order]

        _, first = np.unique(ni_s, return_index=True)
        ni_u  = ni_s[first]
        tot_u = tot_s[first]

        kept = min(bs, len(ni_u))
        top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
        top  = top[np.argsort(tot_u[top])]

        bidx[:kept]  = ni_u[top]
        bcost[:kept] = tot_u[top]
        if kept < bs:
            bidx[kept:]  = bidx[kept - 1]
            bcost[kept:] = np.inf
        bn = kept

        result[step] = tw_tvt[bidx[0]]

    return result


def run_beam_ensemble(hw, tw):
    """Average 14 beam-search configs."""
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    last_tvt = float(kn.iloc[-1]['TVT_input'])
    tw_s  = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr    = gr_all[ev.index]

    beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
                    for (bs, mc, es, r) in BEAM_CONFIGS]

    beam_mean = np.stack(beam_results, 0).mean(0)

    out = hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)] = beam_mean
    return out


def selector_well_code(hw):
    eval_mask = hw['TVT_input'].isna().to_numpy()
    n_eval = float(eval_mask.sum())
    z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
    z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
    n_bin = int(n_eval > SELECTOR_N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
    code = n_bin + 2 * z_bin
    variant = SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
    return code, variant, n_eval, z_span


def parse_selector_variant(name):
    parts = name.split('_')
    scale = float(parts[2])
    beam_weight = 0.0
    hold_weight = 0.0
    if 'beam' in parts:
        beam_weight = float(parts[parts.index('beam') + 1])
    if 'hold' in parts:
        hold_weight = float(parts[parts.index('hold') + 1])
    return scale, beam_weight, hold_weight


def apply_selector_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
    scale, beam_weight, hold_weight = parse_selector_variant(name)
    base = pf_by_scale.get(f"pf_scale_{scale:g}")
    if base is None:
        base = pf_by_scale[SELECTOR_GLOBAL_VARIANT.split('_beam_')[0].split('_hold_')[0]]
    pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
    pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
    return pred


def residual_distance_bucket_code(eval_step):
    bins = np.asarray([49, 249, 999, 2499], dtype=float)
    return np.searchsorted(bins, eval_step, side='right').astype(np.int16)


def find_residual_feature_path():
    local = os.path.join(
        'experiments',
        'exp029_public_sel15_pf_oof_feature_generation',
        'features',
        RESIDUAL_FEATURE_FILE,
    )
    if os.path.exists(local):
        print(f'Residual train feature path={local}')
        return local
    hits = sorted(glob.glob(f'/kaggle/input/**/{RESIDUAL_FEATURE_FILE}', recursive=True))
    if hits:
        print(f'Residual train feature path={hits[0]}')
        return hits[0]
    raise FileNotFoundError(
        f'Cannot locate {RESIDUAL_FEATURE_FILE}; add kentookumura/exp029-sel15-pf-oof-train as a kernel source.'
    )


def sample_residual_training_rows(frame):
    rng = np.random.default_rng(RESIDUAL_RANDOM_SEED)
    eval_step = frame['eval_step'].to_numpy(dtype=float)
    bucket_code = residual_distance_bucket_code(eval_step)
    selected = []
    all_idx = np.arange(len(frame), dtype=np.int64)
    for bucket in np.unique(bucket_code):
        idx = all_idx[bucket_code == bucket]
        if len(idx) > RESIDUAL_MAX_TRAIN_ROWS_PER_BUCKET:
            idx = rng.choice(idx, size=RESIDUAL_MAX_TRAIN_ROWS_PER_BUCKET, replace=False)
        selected.append(idx)
    idx = np.concatenate(selected) if selected else all_idx
    if len(idx) > RESIDUAL_MAX_TRAIN_ROWS:
        idx = rng.choice(idx, size=RESIDUAL_MAX_TRAIN_ROWS, replace=False)
    return np.asarray(np.sort(idx), dtype=np.int64)


def fit_residual_model():
    feature_path = find_residual_feature_path()
    usecols = set(RESIDUAL_FEATURE_COLUMNS + ['target_tvt', 'pf_pred', 'eval_step'])
    frame = pd.read_csv(feature_path, usecols=lambda col: col in usecols)
    missing = sorted(usecols - set(frame.columns))
    if missing:
        raise ValueError(f'Residual feature artifact is missing columns: {missing}')
    frame = frame[np.isfinite(frame['target_tvt'].to_numpy(dtype=float))].reset_index(drop=True)
    train_idx = sample_residual_training_rows(frame)
    x_train = frame.loc[train_idx, RESIDUAL_FEATURE_COLUMNS].to_numpy(dtype=np.float32, copy=True)
    y_train = (
        frame.loc[train_idx, 'target_tvt'].to_numpy(dtype=float)
        - frame.loc[train_idx, 'pf_pred'].to_numpy(dtype=float)
    )
    y_train = np.clip(y_train, -RESIDUAL_TARGET_CLIP, RESIDUAL_TARGET_CLIP)
    model = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', Ridge(alpha=RESIDUAL_RIDGE_ALPHA)),
        ]
    )
    model.fit(x_train, y_train)
    info = {
        'feature_path': feature_path,
        'artifact_rows': int(len(frame)),
        'train_rows': int(len(train_idx)),
        'feature_count': int(len(RESIDUAL_FEATURE_COLUMNS)),
        'candidate': 'ridge_residual_shrink0p5_clip20p0',
    }
    print('Residual model fitted:', json.dumps(info, sort_keys=True))
    return model, info


def run_particle_filter_diag(hw, tw, n_particles=500, seed=42):
    tw_s = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr_raw = tw_s['GR'].values.astype(float)
    tw_gr_mean = float(np.nanmean(tw_gr_raw)) if np.isfinite(np.nanmean(tw_gr_raw)) else 0.0
    tw_gr = pd.Series(tw_gr_raw).interpolate(limit_direction='both').fillna(tw_gr_mean).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    out_vals = hw['TVT_input'].values.astype(float).copy()
    if len(ev) == 0:
        return {
            'prediction': out_vals,
            'log_likelihood': 0.0,
            'mean_effective_particles': float(n_particles),
            'min_effective_particles': float(n_particles),
            'resample_count': 0,
            'gr_sigma': 0.0,
            'initial_rate': 0.0,
        }
    if len(kn) == 0:
        fallback = float(np.nanmean(tw_tvt)) if np.isfinite(np.nanmean(tw_tvt)) else 0.0
        out_vals[list(ev.index)] = fallback
        return {
            'prediction': out_vals,
            'log_likelihood': -1e9,
            'mean_effective_particles': 0.0,
            'min_effective_particles': 0.0,
            'resample_count': 0,
            'gr_sigma': 0.0,
            'initial_rate': 0.0,
        }

    last = kn.iloc[-1]
    last_tvt = float(last['TVT_input'])
    last_Z = float(last['Z'])
    last_MD = float(last['MD'])

    known_gr = kn['GR'].interpolate(limit_direction='both').fillna(tw_gr_mean).values.astype(float)
    tw_at_k = np.interp(kn['TVT_input'].values.astype(float), tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(known_gr - tw_at_k), 10., 60.))

    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    N = int(n_particles)
    rng = np.random.default_rng(seed)
    pos = last_tvt + last_Z + 3.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N

    MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5
    md_v = ev['MD'].values.astype(float)
    z_v = ev['Z'].values.astype(float)
    gr_v = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr_mean).values.astype(float)[ev.index]

    res = np.empty(len(ev), dtype=float)
    eff_values = []
    log_lik = 0.0
    resample_count = 0
    prev_MD = last_MD

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d = (gr_v[i] - eg) / gs
        lk = np.exp(-0.5 * np.minimum(d**2, 600.))
        lk = np.maximum(lk, 1e-300)
        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        n_eff = float(1.0 / (w**2).sum())
        eff_values.append(n_eff)
        if n_eff < RESAMP * N:
            cum = np.cumsum(w)
            u0 = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos = pos[idx] + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w = np.ones(N) / N
            resample_count += 1

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    eff = np.asarray(eff_values, dtype=float)
    return {
        'prediction': out_vals,
        'log_likelihood': float(log_lik),
        'mean_effective_particles': float(eff.mean()) if eff.size else float(N),
        'min_effective_particles': float(eff.min()) if eff.size else float(N),
        'resample_count': int(resample_count),
        'gr_sigma': gs,
        'initial_rate': ir,
    }


def run_pf_lik_ensemble_scales_diag(hw, tw, n_particles=500, n_seeds=128, scales=SELECTOR_SCALES):
    diagnostics = [
        run_particle_filter_diag(hw, tw, n_particles=n_particles, seed=42 + seed)
        for seed in range(int(n_seeds))
    ]
    pred_arr = np.stack([item['prediction'] for item in diagnostics], 0)
    liks = np.asarray([item['log_likelihood'] for item in diagnostics], dtype=float)
    liks_n = liks - np.nanmax(liks)
    predictions_by_scale = {}
    for scale in scales:
        weights = np.exp(liks_n / float(scale))
        weights /= weights.sum()
        predictions_by_scale[f'pf_scale_{scale:g}'] = (weights[:, None] * pred_arr).sum(0)
    return {
        'predictions_by_scale': predictions_by_scale,
        'seed_mean': pred_arr.mean(0),
        'seed_std': pred_arr.std(0),
        'log_likelihoods': liks,
        'diagnostics': diagnostics,
    }


def beam_search_with_cost(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    n = len(hgr)
    nt = len(tw_tvt)
    if n == 0:
        return np.array([last_tvt]), 0.0
    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()
    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))
    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    MC = mc * np.array([2., 1., 0., 1., 2.])
    bidx = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf)
    bcost[0] = 0.
    bn = 1
    result = np.zeros(n)
    for step in range(n):
        gv = sgr[step]
        ni = bidx[:bn, None] + MOVES[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)
        gr_e = (gv - tw_gr[ci])**2 / es
        tot = bcost[:bn, None] + gr_e + MC[None, :]
        tot = np.where(valid, tot, np.inf)
        ni_f = ni.flatten()
        tot_f = tot.flatten()
        vf = valid.flatten()
        ni_f = ni_f[vf]
        tot_f = tot_f[vf]
        order = np.argsort(tot_f)
        ni_s = ni_f[order]
        tot_s = tot_f[order]
        _, first = np.unique(ni_s, return_index=True)
        ni_u = ni_s[first]
        tot_u = tot_s[first]
        kept = min(bs, len(ni_u))
        top = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
        top = top[np.argsort(tot_u[top])]
        bidx[:kept] = ni_u[top]
        bcost[:kept] = tot_u[top]
        if kept < bs:
            bidx[kept:] = bidx[kept - 1]
            bcost[kept:] = np.inf
        bn = kept
        result[step] = tw_tvt[bidx[0]]
    return result, float(bcost[0])


def run_beam_ensemble_diag(hw, tw):
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    out_template = hw['TVT_input'].values.astype(float).copy()
    if len(ev) == 0:
        return {
            'mean': out_template,
            'std': np.zeros_like(out_template),
            'min': out_template,
            'max': out_template,
            'final_cost_mean': 0.0,
            'final_cost_min': 0.0,
        }
    if len(kn) == 0:
        fallback = np.nanmean(out_template) if np.isfinite(np.nanmean(out_template)) else 0.0
        out_template[list(ev.index)] = fallback
        return {
            'mean': out_template,
            'std': np.zeros_like(out_template),
            'min': out_template,
            'max': out_template,
            'final_cost_mean': 0.0,
            'final_cost_min': 0.0,
        }
    last_tvt = float(kn.iloc[-1]['TVT_input'])
    tw_s = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr_raw = tw_s['GR'].values.astype(float)
    tw_gr_mean = float(np.nanmean(tw_gr_raw)) if np.isfinite(np.nanmean(tw_gr_raw)) else 0.0
    tw_gr = pd.Series(tw_gr_raw).interpolate(limit_direction='both').fillna(tw_gr_mean).values.astype(float)
    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr_mean).values.astype(float)
    hgr = gr_all[ev.index]
    members = []
    costs = []
    for bs, mc, es, r in BEAM_CONFIGS:
        eval_pred, cost = beam_search_with_cost(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
        full = out_template.copy()
        full[list(ev.index)] = eval_pred
        members.append(full)
        costs.append(cost)
    arr = np.stack(members, 0)
    costs = np.asarray(costs, dtype=float)
    return {
        'mean': arr.mean(0),
        'std': arr.std(0),
        'min': arr.min(0),
        'max': arr.max(0),
        'final_cost_mean': float(costs.mean()),
        'final_cost_min': float(costs.min()),
    }


def likelihood_summary(log_likelihoods, scale):
    liks = np.asarray(log_likelihoods, dtype=float)
    centered = liks - np.nanmax(liks)
    weights = np.exp(centered / float(scale))
    weights /= weights.sum()
    sorted_weights = np.sort(weights)[::-1]
    entropy = float(-np.sum(weights * np.log(np.maximum(weights, 1e-300))))
    return {
        'pf_lik_best': float(np.nanmax(liks)),
        'pf_lik_mean': float(np.nanmean(liks)),
        'pf_lik_std': float(np.nanstd(liks)),
        'pf_lik_gap_best_second': float(np.sort(liks)[-1] - np.sort(liks)[-2]) if len(liks) >= 2 else 0.0,
        'pf_weight_entropy': entropy,
        'pf_weight_top': float(sorted_weights[0]) if len(sorted_weights) else 1.0,
        'pf_weight_best_second_gap': float(sorted_weights[0] - sorted_weights[1]) if len(sorted_weights) >= 2 else 1.0,
        'pf_effective_seeds': float(1.0 / np.square(weights).sum()),
    }


def build_hidden_residual_features(wid, hw, pf_result, beam_result, tvt_selector, selector_code, selector_variant, selector_n_eval, selector_z_span, last_known_tvt):
    eval_mask = hw['TVT_input'].isna().to_numpy()
    eval_indices = np.flatnonzero(eval_mask).astype(int)
    if len(eval_indices) == 0:
        return pd.DataFrame(columns=RESIDUAL_FEATURE_COLUMNS)
    cutoff_row = int(eval_indices[0])
    prefix_length = int(hw['TVT_input'].notna().sum())
    selected_scale, selector_beam_weight, selector_hold_weight = parse_selector_variant(selector_variant)
    scale_key = f'pf_scale_{selected_scale:g}'
    pf_by_scale = pf_result['predictions_by_scale']
    selected_pf = pf_by_scale.get(scale_key, pf_by_scale['pf_scale_8'])
    eval_step = np.arange(len(eval_indices), dtype=float)
    frame = pd.DataFrame({
        'pf_pred': tvt_selector[eval_indices],
        'last_anchor_tvt': last_known_tvt,
        'beam_pred': beam_result['mean'][eval_indices],
        'pf_selected_scale_pred': selected_pf[eval_indices],
        'pf_scale_3': pf_by_scale['pf_scale_3'][eval_indices],
        'pf_scale_5': pf_by_scale['pf_scale_5'][eval_indices],
        'pf_scale_8': pf_by_scale['pf_scale_8'][eval_indices],
        'pf_scale_12': pf_by_scale['pf_scale_12'][eval_indices],
        'pf_seed_mean': pf_result['seed_mean'][eval_indices],
        'pf_seed_std': pf_result['seed_std'][eval_indices],
        'beam_spread': beam_result['std'][eval_indices],
        'beam_min': beam_result['min'][eval_indices],
        'beam_max': beam_result['max'][eval_indices],
        'beam_final_cost_mean': beam_result['final_cost_mean'],
        'beam_final_cost_min': beam_result['final_cost_min'],
        'selector_code': int(selector_code),
        'selector_scale': selected_scale,
        'selector_beam_weight': selector_beam_weight,
        'selector_hold_weight': selector_hold_weight,
        'selector_n_eval': selector_n_eval,
        'selector_z_span': selector_z_span,
        'pseudo_cutoff_fraction': cutoff_row / max(float(len(hw)), 1.0),
        'cutoff_row': cutoff_row,
        'row_idx': eval_indices,
        'prefix_length': prefix_length,
        'eval_step': eval_step,
        'eval_fraction': eval_step / max(float(len(eval_indices) - 1), 1.0),
        'MD': hw.loc[eval_indices, 'MD'].to_numpy(dtype=float),
        'X': hw.loc[eval_indices, 'X'].to_numpy(dtype=float),
        'Y': hw.loc[eval_indices, 'Y'].to_numpy(dtype=float),
        'Z': hw.loc[eval_indices, 'Z'].to_numpy(dtype=float),
        'GR': hw.loc[eval_indices, 'GR'].to_numpy(dtype=float),
        'gr_isna': hw.loc[eval_indices, 'GR'].isna().astype(int).to_numpy(),
        'gr_prefix_availability': float(hw.loc[:max(cutoff_row - 1, 0), 'GR'].notna().mean()) if cutoff_row > 0 else 0.0,
        'gr_eval_availability': float(hw.loc[eval_indices, 'GR'].notna().mean()),
    })
    for key, value in likelihood_summary(pf_result['log_likelihoods'], selected_scale).items():
        frame[key] = value
    diagnostics = pf_result['diagnostics']
    frame['pf_gr_sigma_mean'] = float(np.mean([item['gr_sigma'] for item in diagnostics]))
    frame['pf_initial_rate_mean'] = float(np.mean([item['initial_rate'] for item in diagnostics]))
    frame['pf_mean_effective_particles'] = float(np.mean([item['mean_effective_particles'] for item in diagnostics]))
    frame['pf_min_effective_particles'] = float(np.min([item['min_effective_particles'] for item in diagnostics]))
    frame['pf_resample_count_mean'] = float(np.mean([item['resample_count'] for item in diagnostics]))
    frame['pf_pred_minus_last_anchor'] = frame['pf_pred'] - frame['last_anchor_tvt']
    frame['pf_beam_diff'] = frame['pf_pred'] - frame['beam_pred']
    frame['abs_pf_beam_diff'] = frame['pf_beam_diff'].abs()
    frame['well_id'] = wid
    return frame[RESIDUAL_FEATURE_COLUMNS + ['well_id']]


def apply_residual_correction(base_pred, residual_features, residual_model):
    corrected = base_pred.copy()
    if residual_features.empty:
        return corrected, np.asarray([], dtype=float), np.asarray([], dtype=int)
    x = residual_features[RESIDUAL_FEATURE_COLUMNS].to_numpy(dtype=np.float32, copy=True)
    residual = residual_model.predict(x)
    residual = np.clip(residual, -RESIDUAL_PRED_CLIP, RESIDUAL_PRED_CLIP)
    row_idx = residual_features['row_idx'].to_numpy(dtype=int)
    corrected[row_idx] = base_pred[row_idx] + RESIDUAL_SHRINK * residual
    return corrected, residual, row_idx


RESIDUAL_MODEL, RESIDUAL_MODEL_INFO = fit_residual_model()

sample = pd.read_csv(os.path.join(INPUT_DIR, 'sample_submission.csv'))
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str[9:].astype(int)

train_wids = set(
    os.path.basename(f).split('__')[0]
    for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))
)
print(f'Training wells available: {len(train_wids)}')

rows = []
audit_rows = []
for wid in TEST_WELLS:
    print(f'\nProcessing {wid}...')
    hw_te, tw_te = load_well(wid, 'test')

    tvt_phys = None
    hw_tr    = None
    tw_tr    = None

    # Physical model for visible wells
    if wid in train_wids:
        try:
            hw_tr, tw_tr = load_well(wid, 'train')
            hw_te['TVT_input'] = hw_tr['TVT_input'].values
            tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
            print(f'  Physical model OK')
        except Exception as e:
            print(f'  Physical model failed: {e}')
            tvt_phys = None

    selector_code, selector_variant, selector_n_eval, selector_z_span = selector_well_code(hw_te)

    # 128-seed likelihood-weighted PF ensemble, cached across selector scales.
    pf_result = None
    try:
        tw_ref = tw_tr if tw_tr is not None else tw_te
        pf_result = run_pf_lik_ensemble_scales_diag(hw_te, tw_ref, n_particles=500, n_seeds=128)
        pf_by_scale = pf_result['predictions_by_scale']
        tvt_pf = pf_by_scale["pf_scale_8"]
        print(f'  PF 128-seed lik-ensemble OK scales={SELECTOR_SCALES}')
    except Exception as e:
        print(f'  PF failed: {e}')
        last_known = hw_te['TVT_input'].dropna()
        last_val   = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
        tvt_pf = hw_te['TVT_input'].fillna(last_val).values.astype(float)
        pf_by_scale = {f"pf_scale_{scale:g}": tvt_pf.copy() for scale in SELECTOR_SCALES}

    try:
        tw_ref = tw_tr if tw_tr is not None else tw_te
        beam_result = run_beam_ensemble_diag(hw_te, tw_ref)
        tvt_beam = beam_result['mean']
        print(f'  Beam 14-config ensemble OK')
    except Exception as e:
        print(f'  Beam failed: {e}')
        tvt_beam = tvt_pf.copy()
        beam_result = {
            'mean': tvt_beam,
            'std': np.zeros_like(tvt_beam),
            'min': tvt_beam,
            'max': tvt_beam,
            'final_cost_mean': 0.0,
            'final_cost_min': 0.0,
        }

    last_known = hw_te['TVT_input'].dropna()
    last_known_tvt = float(last_known.iloc[-1]) if len(last_known) > 0 else float(np.nanmean(tvt_pf))
    tvt_selector = apply_selector_variant(selector_variant, pf_by_scale, tvt_beam, last_known_tvt)
    tvt_corrected = tvt_selector.copy()
    residual_by_row = {}
    if tvt_phys is None and pf_result is not None:
        residual_features = build_hidden_residual_features(
            wid,
            hw_te,
            pf_result,
            beam_result,
            tvt_selector,
            selector_code,
            selector_variant,
            selector_n_eval,
            selector_z_span,
            last_known_tvt,
        )
        tvt_corrected, residual_pred, residual_row_idx = apply_residual_correction(
            tvt_selector,
            residual_features,
            RESIDUAL_MODEL,
        )
        residual_by_row = {int(idx): float(res) for idx, res in zip(residual_row_idx, residual_pred)}
        print(f'  Residual correction OK rows={len(residual_features)}')
    elif tvt_phys is None:
        print('  Residual correction skipped because PF diagnostics failed')
    print(
        f'  Selector code={selector_code} variant={selector_variant} '
        f'n_eval={selector_n_eval:.0f} z_span={selector_z_span:.3f}'
    )

    ws = sample[sample['well'] == wid]
    for _, row in ws.iterrows():
        ridx = int(row['row_idx'])
        if tvt_phys is not None:
            # Visible well: physical model is primary and is not altered by this audit.
            original_val = float(tvt_phys.iloc[ridx])
            tvt_val = original_val
            source = 'physical_visible'
            residual_pred = 0.0
        else:
            # Hidden well: exp032-supported clipped Ridge residual correction on public selector.
            original_val = float(tvt_selector[ridx])
            tvt_val = float(tvt_corrected[ridx])
            source = 'ridge_residual_shrink0p5_clip20p0_hidden'
            residual_pred = float(residual_by_row.get(ridx, 0.0))
        rows.append({'id': row['id'], 'tvt': tvt_val})
        audit_rows.append({
            'id': row['id'],
            'well': wid,
            'row_idx': ridx,
            'source': source,
            'selector_variant': selector_variant,
            'last_known_tvt': last_known_tvt,
            'original_tvt': original_val,
            'corrected_tvt': tvt_val,
            'predicted_residual': residual_pred,
            'diff': tvt_val - original_val,
        })
    print(f'  Added {len(ws)} rows')

submission = pd.DataFrame(rows)
audit = pd.DataFrame(audit_rows)
original_submission = audit[['id', 'original_tvt']].rename(columns={'original_tvt': 'tvt'})
original_submission.to_csv('public_sel15_original_selector_submission.csv', index=False)
audit.to_csv('public_sel15_residual_corrected_diff.csv', index=False)

changed = audit[audit['diff'].abs() > 1e-12].copy()
summary = {
    'experiment': 'exp033_public_sel15_pf_residual_inference_port',
    'base': 'exp027_public_replay_needless090_sel15_spread3',
    'candidate': 'ridge_residual_shrink0p5_clip20p0_hidden',
    'rows': int(len(submission)),
    'changed_rows': int(len(changed)),
    'changed_wells': int(changed['well'].nunique()) if len(changed) else 0,
    'diff_min': float(audit['diff'].min()) if len(audit) else 0.0,
    'diff_max': float(audit['diff'].max()) if len(audit) else 0.0,
    'diff_mean': float(audit['diff'].mean()) if len(audit) else 0.0,
    'diff_abs_mean': float(audit['diff'].abs().mean()) if len(audit) else 0.0,
    'diff_rmse': float(np.sqrt(np.mean(np.square(audit['diff'])))) if len(audit) else 0.0,
    'original_min': float(original_submission['tvt'].min()) if len(original_submission) else 0.0,
    'original_max': float(original_submission['tvt'].max()) if len(original_submission) else 0.0,
    'corrected_min': float(submission['tvt'].min()) if len(submission) else 0.0,
    'corrected_max': float(submission['tvt'].max()) if len(submission) else 0.0,
}
summary['residual_model'] = RESIDUAL_MODEL_INFO
summary['residual_shrink'] = RESIDUAL_SHRINK
summary['residual_clip'] = RESIDUAL_PRED_CLIP
with open('public_sel15_residual_corrected_summary.json', 'w') as fp:
    json.dump(summary, fp, indent=2, sort_keys=True)

submission.to_csv('submission.csv', index=False)
print(f'\nDone: {len(submission)} rows')
print('Residual correction audit summary:')
print(json.dumps(summary, indent=2, sort_keys=True))
print(submission.head())

